# Criteria and methods

> Every criterion, method and policy, with its name and definition

::: {.callout-note appearance="simple" icon=false}
**Reference: look up a fact.** It describes what trufl does, without advising or teaching. Want to know why? Read an [explanation](../explanation/03_round.ipynb). [About the four kinds of page](../explanation/06_documentation.ipynb).
:::

## Criteria

Callbacks for `State`. Each adds one column.

| Callback | Column | Value for an area | Without measurements |
|---|---|---|---|
| `MaxCB()` | `Max` | Highest value | `NaN` |
| `MinCB()` | `Min` | Lowest value | `NaN` |
| `StdCB()` | `Standard Deviation` | Population standard deviation. 0 for one measurement | `NaN` |
| `CountCB()` | `Count` | Number of measurements, blanks included | `NaN`, not 0 |
| `MoranICB(k=5, p_threshold=0.05, min_n=5)` | `Moran.I` | Moran's I if its pseudo p-value is below `p_threshold`, otherwise `NaN`. `NaN` with `min_n` measurements or fewer | `NaN` |
| `PriorCB(fname_raster)` | `Prior` | Mean of the raster cells whose centre is in the area, all bands pooled | Computed as usual |

Each callback takes a `name` that replaces the column name.

:::{.callout-warning}
`MoranICB` reports significant spatial trends in random data.
:::

## Directions

`is_benefit_x` holds one flag per column, by position.

| Flag | Name | Most urgent value |
|---|---|---|
| `True` | Benefit | Highest |
| `False` | Cost | Lowest |

## Normalization: `n_method`

For a value $x$ of a criterion. Names are not case-sensitive.

| Name | Benefit | Cost | Flags afterwards | Refuses |
|---|---|---|---|---|
| `LINEAR1` | $x / \max$ | $\min / x$ | All benefit | Negative values, a benefit with a maximum of 0, a cost with a minimum of 0 |
| `LINEAR2` | $(x - \min) / (\max - \min)$ | $(\max - x) / (\max - \min)$ | All benefit | A constant criterion |
| `LINEAR3` | $x / \sum x$ | $x / \sum x$ | Unchanged | Negative values, a sum of 0 |
| `VECTOR` | $x / \sqrt{\sum x^2}$ | $x / \sqrt{\sum x^2}$ | Unchanged | Negative values, a norm of 0 |
| `None` | $x$ | $x$ | Unchanged | Values outside 0 to 1 |

After normalization, `get_rank` replaces a missing value with 1 for a benefit and 0 for a cost.

## Weights: `w_vector` or `w_method`

`w_vector` holds one weight per column, by position, adding up to 1. When it is `None`, `w_method` computes the weights.

| `w_method` | Weight of a criterion is proportional to | `c_method` |
|---|---|---|
| `MW` | 1 | Not used |
| `SD` | Its standard deviation | Not used |
| `EM` | 1 minus its entropy. Needs columns that each sum to 1 | Not used |
| `CRITIC` | Its standard deviation times the sum of 1 minus its correlations | `PEARSON` (default), `ABSPEARSON`, `DCOR` |
| `VIC` | Its standard deviation divided by the sum of its correlations | `DCOR` (default), `ABSPEARSON` |

## Scoring: `s_method`

$z_{ij}$ is the normalized value of area $i$ for criterion $j$, and $w_j$ the weight. $d^+_i$ and $d^-_i$ are the Euclidean distances of the weighted values $w_j z_{ij}$ to the most and to the least urgent weighted value of each criterion among the areas.

| Name | Score | Rank 1 |
|---|---|---|
| `CP` | $d^+_i$ | Lowest score |
| `TOPSIS` | $d^-_i / (d^+_i + d^-_i)$ | Highest score |

Equal scores receive different ranks, in arbitrary order.

## Allocation: `policy`

For a budget $B$, $N$ areas and rank $r$. Names are case-sensitive.

| Name | Samples for an area | Rounding |
|---|---|---|
| `Weighted` (default) | $B \cdot (1/r) / \sum (1/r)$ | Nearest integer |
| `quantiles` | $B \cdot p / (N/4)$, with $p$ = 0.5, 0.3, 0.2 and 0 for the four quartiles of rank | Down |

Values below `min` are then raised to `min`. Nothing is taken from other areas, so the total can differ from $B$.

Totals for 100 areas:

In [ ]:
#| hide
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd
from fastcore.test import test_eq

from trufl.utils import gridder
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, PriorCB
from trufl.optimizer import Optimizer
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector

In [ ]:
#| echo: false
every_rank = np.arange(1, 101)
budgets = pd.Index([20, 100, 150, 400, 600, 1000], name='budget')
pd.DataFrame({f'{policy}, min={m}': [rank_to_sample(every_rank, b, m, policy).sum() for b in budgets]
              for policy in ('quantiles', 'Weighted') for m in (0, 1)}, index=budgets)

,"quantiles, min=0","quantiles, min=1","Weighted, min=0","Weighted, min=1"
budget,,,,
20,0,100,11,104
100,75,125,86,148
150,125,150,142,185
400,375,400,402,402
600,575,600,600,600
1000,1000,1025,999,999


## Placement: `Sampler.sample`

| Argument | Accepted | Meaning |
|---|---|---|
| `n` | NumPy integer array | Points per area, in the row order of the areas |
| `method` | `'uniform'` (default), or a name from `pointpats.random` | How points are drawn |
| `rng` | NumPy `Generator` | One stream of random numbers for all areas |
| `rng` | Integer | The stream restarts in each area |

In [ ]:
#| hide
x = pd.DataFrame({'a': [1., 2., 4.], 'b': [1., 3., 2.]})
from trufl.mcdm import normalize, score, weigh
z, flags = normalize(x.to_numpy(), [True, False], 'linear1')
test_eq(z.tolist(), [[.25, 1.], [.5, 1/3], [1., .5]]); test_eq(flags, [True, True])
test_eq(normalize(x.to_numpy(), [True, False], 'LINEAR3')[1], [True, False])
w = np.array([.5, .5])
test_eq(np.allclose(score(z, flags, w, 'CP')[0], np.sqrt((((z.max(0) - z) * w) ** 2).sum(1))), True)
test_eq(np.allclose(weigh(z, 'SD'), z.std(0) / z.std(0).sum()), True)
test_eq(rank_to_sample(np.arange(1, 9), 80, policy='quantiles'), [20, 20, 12, 12, 8, 8, 0, 0])
w8 = 1 / np.arange(1, 9)
test_eq(rank_to_sample(np.arange(1, 9), 80), np.round(80 * w8 / w8.sum()).astype(int))
test_eq([cb.name for cb in (MaxCB(), MinCB(), StdCB(), CountCB(), PriorCB(''))], ['Max', 'Min', 'Standard Deviation', 'Count', 'Prior'])